In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch
from imrw import imread

from src.inference.predictor import Predictor

In [ ]:
candidates = sorted(
    d
    for d in (REPO_ROOT / "run").glob("*")
    if (d / "config.yaml").is_file() and (d / "weights" / "generator.pth").is_file()
)
assert candidates, "no trained run found — run `python run_train.py` first"
run_dir = str(candidates[-1])
device = "cuda" if torch.cuda.is_available() else "cpu"

predictor = Predictor(run_dir=run_dir, device=device)
print("run_dir     :", run_dir)
print("device      :", device)
print("train_shape :", predictor.train_shape)
print("in_channels :", predictor.in_channels)

In [ ]:
def center_crop(img, h, w):
    ih, iw = img.shape[:2]
    y0 = (ih - h) // 2
    x0 = (iw - w) // 2
    return img[y0 : y0 + h, x0 : x0 + w]


D, H, W = predictor.train_shape

anchor_paths = sorted((REPO_ROOT / "data" / "default" / "2d").glob("*.png"))[:2]
anchor_images = [center_crop(imread(str(p)), H, W) for p in anchor_paths]
anchor_indices = [8, 40]

# Predictor returns (D, H, W, C); drop C=1 for grayscale
raw = predictor.predict(
    anchor_images=anchor_images,
    anchor_indices=anchor_indices,
    seed=0,
)
volume = raw[..., 0] if predictor.in_channels == 1 else raw

print("anchor paths   :", [p.name for p in anchor_paths])
print("anchor shape   :", anchor_images[0].shape)
print("anchor dtype   :", anchor_images[0].dtype)
print("anchor indices :", anchor_indices)
print("volume shape   :", volume.shape)
print("volume range   :", f"[{volume.min():+.3f}, {volume.max():+.3f}]")

In [ ]:
def to_display(img):
    return np.clip((img + 1.0) / 2.0, 0.0, 1.0)


D_axis = volume.shape[0]
indices = sorted(
    {0, D_axis // 4, D_axis // 2, 3 * D_axis // 4, D_axis - 1} | set(anchor_indices)
)

fig, axes = plt.subplots(1, len(indices), figsize=(2 * len(indices), 2.4))
cmap = "gray" if predictor.in_channels == 1 else None
for ax, k in zip(axes, indices):
    ax.imshow(to_display(volume[k]), cmap=cmap, vmin=0.0, vmax=1.0)
    title = f"k={k}" + (" (anchor)" if k in anchor_indices else "")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("generated slices along axis 0", y=1.02)
plt.tight_layout()
plt.show()